In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu

np.random.seed(42)
import random
random.seed(42)

In [2]:
mdata = mu.read("./data/EAE/CCA/merged_EAE_CD4_ccaHigh_in_cloned.h5mu")
weights = np.load("./data/EAE/CCA/merged_EAE_CD4_cca_weights.npz")
gex_weights = weights["gex"]
tcr_weights = weights["tcr"]

In [3]:
print(gex_weights.shape)
print(tcr_weights.shape)

(50, 4)
(1317, 4)


In [4]:
mdata

MuData object with n_obs × n_vars = 23770 × 4000
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'condition', 'sample_id', 'state', 'VDJ_1_cdr3_aa_length', 'VJ_1_cdr3_aa_length', 'set'
  obsm:	'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call', 'X_VDJ_1_cdr3_aa_atchley', 'X_VDJ_1_cdr3_aa_atchley_pairwise', 'X_VDJ_1_cdr3_aa_composition', 'X_VJ_1_cdr3_aa_atchley', 'X_VJ_1_cdr3_aa_atchley_pairwise', 'X_VJ_1_cdr3_aa_composition', 'tcr_cca', 'tcr_embs'
  2 modalities
    gex:	23770 x 4000
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE', 'Tissue_group', 'CV_score_0', 'CV_score_1', 'CV_score_2', 'CV_score_3', 'CV_score_0_high', 'CV_score_1_high', 'CV_score_2_high', 'CV_score_3_high', 'TCR_logit_score'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'X_umap_harmony', 'cell_type_colors', 'hvg', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'rank_genes_groups', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'
    airr:	23770 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition', 'GSE'
      obsm:	'airr', 'chain_indices'

In [5]:
print(mdata.obsm['gex_cca'].shape)
print(mdata.obsm['tcr_cca'].shape)

(23770, 4)
(23770, 4)


In [6]:
TCR_emb = mdata.obsm['tcr_embs']
cv_dim = 1
weighted_TCR = TCR_emb * tcr_weights[:, cv_dim]
weighted_TCR.shape

(23770, 1317)

In [7]:
print(mdata['gex'].obsm['X_pca_harmony'].shape)


(23770, 50)


In [ ]:
f_list = [mdata['gex'].obsm['X_pca_harmony'], weighted_TCR]
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
for i in range(len(f_list)):
    f_list[i] = scaler.fit_transform(f_list[i])

features = np.concatenate(f_list, axis=1)


array([[ 0.62560376,  1.10133572, -0.87253336, ...,  0.        ,
        -0.35844659, -1.05877238],
       [-1.63332754, -0.01652227,  1.18638418, ...,  0.        ,
         0.60746151, -0.0880847 ],
       [-0.02219056,  1.0785707 , -0.94302064, ...,  0.        ,
         0.60746151,  0.88260298],
       ...,
       [-0.65729078,  1.16957516, -1.09955649, ...,  0.        ,
        -0.35844659, -1.05877238],
       [ 0.40129945, -0.9328583 ,  0.49275404, ...,  0.        ,
        -1.32435469,  0.88260298],
       [ 0.30911404, -0.39253857,  0.02236864, ...,  0.        ,
        -1.32435469, -0.0880847 ]], shape=(23770, 1367))

In [18]:
labels = mdata['gex'].obs[['tissue', 'cell_type', 'state']]
labels = pd.concat([labels, mdata.obs['set']], axis=1)
labels.columns = [f"label_{col}" for col in labels.columns]

labels

,label_tissue,label_cell_type,label_state,label_set
ACCAGAGAGCCAGAGA-1_0516_CNS,CNS,CD4,Activation,train
ATGAGGAGTCTATTCG-1_0516_CNS,CNS,NaN,Activation,train
CACTACGCACCACACA-1_0516_CNS,CNS,Treg,Activation,train
CAGCATATCCCGTAAA-1_0516_CNS,CNS,CD4,Activation,train
CCCAACACAGCGTATT-1_0516_CNS,CNS,NaN,Activation,train
...,...,...,...,...
TTTGTCAAGGTACTCT-1-b7m2,CNS,NaN,NaN,train
TTTGTCAGTAAGTAGT-1-b7m2,SI,Th17,IFN_stim,train
TTTGTCAGTCAAGCGA-1-b7m2,NaN,NaN,NaN,train
TTTGTCAGTGATGCCC-1-b7m2,NaN,NaN,NaN,train


In [19]:
feature_columns = [
    f"feature_gex{i + 1}" for i in range(f_list[0].shape[1])
] + [
    f"feature_tcr{i + 1}" for i in range(f_list[1].shape[1])
]
index = mdata.obs_names
features_df = pd.DataFrame(features, columns=feature_columns, index=index)

combined_df = pd.concat([features_df, labels], axis=1)
combined_df.fillna(0, inplace=True)
combined_df.head()


TypeError: Cannot setitem on a Categorical with a new category (0), set the categories first

In [13]:
combined_df.to_csv("merged_EAE_CD4.csv", index=False)